# RoSE on ChartQAPro — Complete Notebook

| | |
|---|---|
| **Dataset** | [ahmed-masry/ChartQAPro](https://huggingface.co/datasets/ahmed-masry/ChartQAPro) |
| **Model** | `Qwen/Qwen2.5-VL-3B-Instruct` |
| **Targets** | Factoid · Multi Choice · Hypothetical |
| **Output** | Compatible with official `evaluate_predictions.py` |

## How to use (3-teammate split)
| Teammate | `ASSIGNED_TYPE` |
|---|---|
| A | `"Factoid"` |
| B | `"Multi Choice"` |
| C | `"Hypothetical"` |

> The RoSE pool is shared across all types (the pool sees every question in order). Each teammate runs the full dataset but only saves results for their assigned type.


## CELL 1 — GOOGLE DRIVE MOUNT  (run FIRST every session)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/RoSE_ChartQAPro'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")


## CELL 2 — INSTALL PACKAGES  (run once; skip on resume)


In [ ]:
import importlib
if not importlib.util.find_spec('transformers'):
    import subprocess
    subprocess.run([
        "pip", "install", "-q",
        "transformers==4.47.0", "accelerate", "bitsandbytes",
        "sentence-transformers", "qwen-vl-utils",
        "datasets", "Pillow", "anls", "pandas"
    ])
    print("✓ Packages installed")
else:
    print("✓ Already installed")

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## CELL 3 — CONFIG  ← EACH TEAMMATE CHANGES ONLY THIS CELL


In [ ]:

# ┌─────────────────────────────────────────────────────────────┐
# │  TEAMMATE A → "Factoid"                                     │
# │  TEAMMATE B → "Multi Choice"                                │
# │  TEAMMATE C → "Hypothetical"                                │
# └─────────────────────────────────────────────────────────────┘
ASSIGNED_TYPE = "Factoid"   # ← CHANGE THIS FOR EACH TEAMMATE

# Paths (auto-named per teammate so files don't collide)
type_slug     = ASSIGNED_TYPE.lower().replace(" ", "_")
CHECKPOINT    = f"{WORK_DIR}/checkpoint_{type_slug}.json"
OUTPUT_FILE   = f"{WORK_DIR}/results_{type_slug}.json"

# RoSE paper hyperparameters
M_PATHS           = 3      # paper uses 20; 3 fits free T4
K_DEMOS           = 3      # demonstrations per question
LAMBDA_THRESH     = 1.2    # uncertainty threshold multiplier
TEMPERATURE       = 0.7    # diversity temperature
MAX_NEW_TOKENS    = 256
CHECKPOINT_EVERY  = 20     # save every N questions

print(f"\nAssigned type : {ASSIGNED_TYPE}")
print(f"Checkpoint    : {CHECKPOINT}")
print(f"Output file   : {OUTPUT_FILE}")


## CELL 4 — LOAD DATASET FROM HUGGINGFACE


In [ ]:
from datasets import load_dataset
from collections import Counter

print("Loading ChartQAPro from HuggingFace...")
ds = load_dataset("ahmed-masry/ChartQAPro", split="test")
data = [dict(row) for row in ds]

print(f"Total questions: {len(data)}")
type_counts = Counter(d["Question Type"] for d in data)
print("Question type distribution:")
for qt, n in type_counts.most_common():
    print(f"  {qt:<20} {n}")

# Filter for assigned type only (for running inference)
assigned_data = [d for d in data if d["Question Type"] == ASSIGNED_TYPE]
print(f"\nThis teammate will infer on: {len(assigned_data)} questions ({ASSIGNED_TYPE})")


## CELL 5 — UNDERSTAND THE DATA FORMAT


In [ ]:
# This cell shows you exactly what each field looks like
# so you understand what you're working with.

print("=== Sample record from dataset ===\n")
sample = assigned_data[0]
for key, val in sample.items():
    if key == "image":
        print(f"  image: <PIL Image {val.size}>")
    else:
        print(f"  {key}: {repr(val)}")

# Key fields:
#   "Question"      → the question text
#   "Answer"        → list e.g. ["2016"] or ["42.5"]
#   "Question Type" → "Factoid", "Multi Choice", "Hypothetical", etc.
#   "Year"          → list of "YES"/"NO" flags (per answer element)
#   "image"         → PIL.Image object (already loaded!)
#   "Choices"       → list of options (for Multi Choice only)


## CELL 6 — LOAD MODEL


In [ ]:
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f"Loading {MODEL_NAME}...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ Model loaded  ({used:.1f}/{total:.1f} GB VRAM used)")


## CELL 7 — LOAD EMBEDDING MODEL (for RoSE similarity)


In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading sentence embedder...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✓ Embedder ready")


## CELL 8 — CORE FUNCTIONS (RoSE algorithm, paper-faithful)


In [ ]:
import math, json, re
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image


# ── 8a. VLM call ────────────────────────────────────────────────

def call_vlm(image, prompt_text, temperature=TEMPERATURE):
    """Single forward pass through Qwen2.5-VL with a PIL image."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image",  "image": image},
            {"type": "text",   "text":  prompt_text},
        ],
    }]
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_input], images=[image], return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=temperature,
            do_sample=(temperature > 0),
            top_p=0.9,
        )
    new = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(new, skip_special_tokens=True).strip()


# ── 8b. Prompts ──────────────────────────────────────────────────

def build_zero_shot_prompt(question, qtype, choices=None):
    """Zero-Shot-CoT prompt (used until pool has k items)."""
    if qtype == "Multi Choice" and choices:
        opts = "\n".join(f"({chr(65+i)}) {c}" for i, c in enumerate(choices))
        return (
            "Look at the chart carefully.\n\n"
            f"Question: {question}\n\nOptions:\n{opts}\n\n"
            "Think step by step, then give the letter of your final answer."
        )
    if qtype == "Hypothetical":
        return (
            "Look at the chart carefully.\n\n"
            f"Hypothetical question: {question}\n\n"
            "Think step by step. Reason about what would change, "
            "then give your final answer."
        )
    # Factoid (and default)
    return (
        "Look at the chart carefully.\n\n"
        f"Question: {question}\n\n"
        "Think step by step, then state your final answer concisely."
    )


def build_few_shot_prompt(question, qtype, demonstrations, choices=None):
    """
    Few-Shot-CoT prompt — paper Eq. 9:
    LLM(q1, r1, a1, ..., qk, rk, ak, qt)
    """
    header = (
        "You are an expert at reading charts. "
        "Here are example questions with step-by-step reasoning:\n\n"
    )
    examples = ""
    for i, d in enumerate(demonstrations, 1):
        examples += (
            f"[Example {i}]\n"
            f"Q: {d['question']}\n"
            f"Reasoning: {d['rationale']}\n"
            f"A: {d['answer']}\n\n"
        )

    if qtype == "Multi Choice" and choices:
        opts = "\n".join(f"({chr(65+i)}) {c}" for i, c in enumerate(choices))
        task = (
            "Now answer the question about the chart above:\n\n"
            f"Q: {question}\n\nOptions:\n{opts}\n\n"
            "Think step by step, then give the letter of your final answer."
        )
    elif qtype == "Hypothetical":
        task = (
            "Now answer the hypothetical question about the chart above:\n\n"
            f"Q: {question}\n\n"
            "Reason about what would change, then give your final answer."
        )
    else:
        task = (
            "Now answer the question about the chart above:\n\n"
            f"Q: {question}\n\n"
            "Think step by step, then state your final answer concisely."
        )
    return header + examples + task


# ── 8c. Answer extraction ────────────────────────────────────────

def extract_final_answer(raw, qtype):
    """
    Pull the answer from chain-of-thought output.
    For Multi Choice: returns just the letter (a/b/c/d/e).
    For others: returns the clean answer string.
    """
    text = raw.strip()
    # Look for explicit markers (last occurrence wins)
    markers = [
        "final answer:", "the answer is", "answer:", "therefore,",
        "thus,", "so the answer", "correct answer is", "answer is",
    ]
    for marker in markers:
        idx = text.lower().rfind(marker)
        if idx != -1:
            tail = text[idx + len(marker):].strip()
            answer = tail.split("\n")[0].split(".")[0].strip()
            if answer:
                if qtype == "Multi Choice":
                    # Extract just the letter
                    m = re.search(r'\b([a-eA-E])\b', answer)
                    if m:
                        return m.group(1).lower()
                return answer

    # MCQ fallback: scan for lone letter
    if qtype == "Multi Choice":
        for line in reversed(text.split("\n")):
            line = line.strip().strip("().,")
            if line.lower() in {"a", "b", "c", "d", "e"}:
                return line.lower()

    # General fallback: last non-empty line
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return lines[-1] if lines else text


# ── 8d. Step counter (paper Eq. 4–5) ────────────────────────────

def count_steps(text):
    """Count reasoning lines. Paper: 'we see a line as one reasoning step'."""
    indicators = [
        "step", "first", "second", "third", "because", "since",
        "therefore", "thus", "so,", "looking at", "chart shows",
        "we can see", "observe", "=", "+", "-", "×", "/"
    ]
    count = sum(
        1 for line in text.split("\n")
        if line.strip() and
        any(ind in line.lower() for ind in indicators)
    )
    return max(count, 1)


# ── 8e. Self-consistency + uncertainty (paper Eq. 1–3) ──────────

def run_m_paths(image, prompt, qtype):
    """
    Generate M_PATHS reasoning paths. Returns:
      majority_answer  (str)
      best_rationale   (str)  — most-step path (Eq. 5)
      uncertainty      (float) — Shannon entropy, normalised
      complexity       (float) — avg steps of majority paths (Eq. 4)
    """
    raw_outputs, answers = [], []
    for _ in range(M_PATHS):
        raw = call_vlm(image, prompt, TEMPERATURE)
        ans = extract_final_answer(raw, qtype)
        raw_outputs.append(raw)
        answers.append(ans.lower().strip())

    counts  = Counter(answers)
    total   = len(answers)
    probs   = [c / total for c in counts.values()]

    # Shannon entropy (Eq. 3), normalised
    entropy     = -sum(p * math.log(p + 1e-12) for p in probs)
    uncertainty = entropy / (math.log(M_PATHS) + 1e-12)

    majority_answer = counts.most_common(1)[0][0]
    majority_paths  = [
        r for r, a in zip(raw_outputs, answers)
        if a == majority_answer
    ]

    # Complexity = avg steps of majority paths (Eq. 4)
    steps      = [count_steps(r) for r in majority_paths]
    complexity = sum(steps) / len(steps)

    # Best rationale = path with most steps (Eq. 5)
    best_rationale = max(majority_paths, key=count_steps)

    return majority_answer, best_rationale, uncertainty, complexity


# ── 8f. Experience Pool (paper §3.1 + Algorithm 1) ──────────────

class ExperiencePool:
    def __init__(self):
        self.pool       = []
        self._emb_matrix = None

    def add(self, question, rationale, answer, qtype,
            uncertainty, complexity):
        emb = embedder.encode(question, convert_to_numpy=True)
        self.pool.append({
            "question":    question,
            "rationale":   rationale,
            "answer":      answer,
            "qtype":       qtype,
            "uncertainty": uncertainty,
            "complexity":  complexity,
            "embedding":   emb,
        })
        self._emb_matrix = np.stack([e["embedding"] for e in self.pool])

    def size(self): return len(self.pool)

    def _partition(self, sim_sorted_idx, k):
        """Algorithm 1: uniform partition → split largest if empty."""
        n = len(sim_sorted_idx)
        if n == 0: return []
        bsize   = n // k
        buckets = []
        for b in range(k):
            s = b * bsize
            e = s + bsize if b < k - 1 else n
            buckets.append(list(sim_sorted_idx[s:e]))
        buckets = [b for b in buckets if b]
        while len(buckets) < k:
            biggest = max(range(len(buckets)), key=lambda i: len(buckets[i]))
            b = buckets.pop(biggest)
            mid = len(b) // 2
            buckets += [b[:mid], b[mid:]]
            buckets  = [b for b in buckets if b]
        return buckets

    def orchestrate(self, question):
        """
        Full RoSE orchestration (Diversity → Uncertainty → Complexity).
        Returns list of K_DEMOS demonstration dicts.
        """
        if self.size() < K_DEMOS:
            return []

        q_emb = embedder.encode(question, convert_to_numpy=True)
        sims  = self._emb_matrix.dot(q_emb) / (
            np.linalg.norm(self._emb_matrix, axis=1) *
            np.linalg.norm(q_emb) + 1e-12
        )
        n_candidates = min(self.size(), 3 * K_DEMOS)
        sorted_idx   = np.argsort(sims)[:n_candidates].tolist()  # low → high

        buckets = self._partition(sorted_idx, K_DEMOS)
        selected = []
        for bucket in buckets[:K_DEMOS]:
            # Uncertainty filter (Eq. 6–7): keep u ≤ λ × u_min
            u_vals   = [self.pool[i]["uncertainty"] for i in bucket]
            u_min    = min(u_vals)
            thresh   = LAMBDA_THRESH * u_min
            filtered = [i for i in bucket
                        if self.pool[i]["uncertainty"] <= thresh]
            if not filtered:
                filtered = [min(bucket,
                               key=lambda i: self.pool[i]["uncertainty"])]
            # Complexity selection (Eq. 8)
            best = max(filtered, key=lambda i: self.pool[i]["complexity"])
            selected.append(self.pool[best])

        return selected[:K_DEMOS]


## CELL 9 — MAIN INFERENCE LOOP


In [ ]:
import time

# ── CSV helper ──────────────────────────────────────────────────

def save_csv(results, csv_path):
    """
    Save results as a clean CSV.
    Columns visible to teammates and professor:
      question | question_type | ground_truth | prediction |
      is_correct_rough | method | uncertainty | complexity | pool_size
    """
    import pandas as pd

    rows = []
    for r in results:
        gt_str   = r["Answer"][-1] if isinstance(r["Answer"], list) else str(r["Answer"])
        pred_str = r.get("prediction", "")
        rough_ok = pred_str.lower().strip() in gt_str.lower().strip() or \
                   gt_str.lower().strip() in pred_str.lower().strip()
        rows.append({
            "question":       r.get("Question", ""),
            "question_type":  r.get("Question Type", ""),
            "ground_truth":   gt_str,
            "prediction":     pred_str,
            "is_correct_rough": rough_ok,
            "method":         r.get("_method", ""),
            "uncertainty":    r.get("_uncertainty", ""),
            "complexity":     r.get("_complexity", ""),
            "pool_size":      r.get("_pool_size", ""),
            "rationale":      r.get("_rationale", "")[:200],  # truncated for readability
        })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")  # utf-8-sig for Excel compatibility
    return df


def run_inference():
    Path(WORK_DIR).mkdir(exist_ok=True)

    # ── Resume from checkpoint ──
    results   = []
    start_idx = 0
    if Path(CHECKPOINT).exists():
        with open(CHECKPOINT) as f:
            results = json.load(f)
        start_idx = len(results)
        print(f"↩  Resuming from checkpoint: sample {start_idx}/{len(assigned_data)}")

    # ── Initialise pool ──
    pool = ExperiencePool()
    for r in results:
        pool.add(
            question    = r["Question"],
            rationale   = r.get("_rationale", r["prediction"]),
            answer      = r["prediction"],
            qtype       = r["Question Type"],
            uncertainty = r.get("_uncertainty", 0.5),
            complexity  = r.get("_complexity",  1.0),
        )

    print(f"\n{'='*60}")
    print(f"  RoSE → {ASSIGNED_TYPE}")
    print(f"  Questions: {len(assigned_data)} | Paths/Q: {M_PATHS} | k: {K_DEMOS}")
    print(f"{'='*60}\n")

    for i, sample in enumerate(assigned_data[start_idx:], start=start_idx):
        qtype   = sample["Question Type"]
        question = sample["Question"]
        image    = sample["image"]        # already a PIL.Image from HuggingFace
        choices  = sample.get("Choices", None)

        # ── Build prompt ──
        method = "zero_shot"
        if pool.size() >= K_DEMOS:
            demos   = pool.orchestrate(question)
            prompt  = build_few_shot_prompt(question, qtype, demos, choices)
            method  = "rose"
        else:
            demos  = []
            prompt = build_zero_shot_prompt(question, qtype, choices)

        # ── Self-consistency inference ──
        t0 = time.time()
        try:
            maj_ans, best_rationale, uncertainty, complexity = \
                run_m_paths(image, prompt, qtype)
        except Exception as e:
            print(f"  ⚠ Error at {i}: {e}")
            maj_ans        = "ERROR"
            best_rationale = ""
            uncertainty    = 1.0
            complexity     = 0.0

        elapsed = time.time() - t0

        # ── Add to pool ──
        pool.add(
            question    = question,
            rationale   = best_rationale,
            answer      = maj_ans,
            qtype       = qtype,
            uncertainty = uncertainty,
            complexity  = complexity,
        )

        # ── Build output record ──
        # MUST match the format expected by evaluate_predictions.py:
        #   Answer, Question Type, Year, prediction
        record = {
            "Question":      question,
            "Answer":        sample["Answer"],        # ground truth list
            "Question Type": qtype,
            "Year":          sample["Year"],           # list of YES/NO
            "prediction":    maj_ans,
            # Private fields (for resume + analysis, not used by eval script)
            "_method":       method,
            "_uncertainty":  round(uncertainty, 4),
            "_complexity":   round(complexity,  4),
            "_rationale":    best_rationale[:300],     # truncated for storage
            "_pool_size":    pool.size(),
        }
        results.append(record)

        # ── Log ──
        gt_str = str(sample["Answer"])
        mark   = "✓" if maj_ans.lower() in gt_str.lower() else "~"
        print(f"[{i+1:04d}/{len(assigned_data)}] {mark} {method.upper():<8} "
              f"pred='{maj_ans[:40]}'  gt={gt_str[:30]}  ({elapsed:.1f}s)")

        # ── Checkpoint ──
        if (i + 1) % CHECKPOINT_EVERY == 0:
            with open(CHECKPOINT, "w") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            save_csv(results, OUTPUT_FILE.replace(".json", "_checkpoint.csv"))
            done = i + 1
            acc  = sum(
                1 for r in results
                if r["prediction"].lower() in str(r["Answer"]).lower()
            ) / done * 100
            print(f"\n  💾 Checkpoint saved ({done}/{len(assigned_data)}) "
                  f"rough-acc={acc:.1f}%\n")

    # ── Final save: JSON (required for eval) + CSV (human-readable) ──
    with open(OUTPUT_FILE, "w") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    csv_path = OUTPUT_FILE.replace(".json", ".csv")
    save_csv(results, csv_path)

    print(f"\n✓ JSON saved → {OUTPUT_FILE}  (used by evaluate_predictions.py)")
    print(f"✓ CSV saved  → {csv_path}       (human-readable, share with team)")
    print(f"  Total records: {len(results)}")
    return results


# ── Run ──
results = run_inference()


## CELL 10 — EVALUATE WITH OFFICIAL SCRIPT


In [ ]:
# This replicates evaluate_predictions.py inline so you don't
# need to download the script separately.

import ast
from anls import anls_score

def fix_list_format(item):
    if not isinstance(item, str): return item
    match = re.match(r"^\[(.*)\]$", item.strip())
    if not match: return item
    content = match.group(1)
    corrected = re.sub(r"(?<!['\w])(\w[^,]*?)(?!['\w])", r"'\1'", content)
    try:    return ast.literal_eval(f"[{corrected}]")
    except: return item

def parse_to_list(text):
    if not isinstance(text, str): return None
    try:
        parsed = ast.literal_eval(text)
    except: return None
    if isinstance(parsed, list):
        return [str(x).strip(" '") for x in parsed]
    return None

def to_float(text):
    try:    return float(text.strip().strip('%'))
    except: return None

def evaluate_single_answer(target, prediction, max_relative_change=0.05):
    t = target.strip().strip('%')
    p = prediction.strip().strip('%')
    t_f, p_f = to_float(t), to_float(p)
    if t_f is not None and p_f is not None:
        if t_f == 0.0: return 1.0 if p_f == 0.0 else 0.0
        return 1.0 if abs(p_f - t_f) / abs(t_f) <= max_relative_change else 0.0
    return anls_score(prediction=p.lower(), gold_labels=[t.lower()], threshold=0.5)

def relaxed_correctness(target, prediction, year_flags, always_exact=False):
    fixed_t = fix_list_format(target)
    t_list  = parse_to_list(str(fixed_t)) or [str(target)]
    p_list  = parse_to_list(str(prediction)) or [str(prediction)]
    if year_flags and len(year_flags) < len(t_list):
        year_flags = year_flags * len(t_list)
    scores = []
    for idx in range(max(len(t_list), len(p_list))):
        if idx >= len(t_list) or idx >= len(p_list):
            scores.append(0.0); continue
        t_item, p_item = t_list[idx], p_list[idx]
        flag = (year_flags[idx].upper() == 'YES') if year_flags else False
        if flag or always_exact:
            scores.append(1.0 if t_item.strip().lower() == p_item.strip().lower() else 0.0)
        else:
            scores.append(evaluate_single_answer(t_item, p_item))
    return sum(scores) / len(scores) if scores else 0.0

def official_evaluate(results):
    """Run the same evaluation as evaluate_predictions.py"""
    scores_by_type = defaultdict(list)
    all_scores     = []

    for r in results:
        gt      = r["Answer"][-1].strip(".").strip("\n")
        pred    = r["prediction"].strip(".").strip("\n")
        qtype   = r["Question Type"]
        yflags  = r["Year"]

        if qtype == "Conversational":
            yflags = yflags[-1:]

        always_exact = qtype in ["Fact Checking", "Multi Choice"]
        score = relaxed_correctness(gt, pred, yflags, always_exact)
        scores_by_type[qtype].append(score)
        all_scores.append(score)

    print("\n" + "="*50)
    print("  OFFICIAL EVALUATION RESULTS")
    print("="*50)
    for qtype, scores in sorted(scores_by_type.items()):
        acc = sum(scores) / len(scores) * 100
        print(f"  {qtype:<20}  {acc:6.2f}%  (n={len(scores)})")
    overall = sum(all_scores) / len(all_scores) * 100 if all_scores else 0
    print(f"  {'Overall':<20}  {overall:6.2f}%")
    print("="*50)
    return {k: sum(v)/len(v) for k, v in scores_by_type.items()} | {"Overall": overall/100}

scores = official_evaluate(results)

# ── Quick CSV preview (your results at a glance) ──
import pandas as pd

csv_path = OUTPUT_FILE.replace(".json", ".csv")
if Path(csv_path).exists():
    df = pd.read_csv(csv_path)
    print(f"\n── Your CSV: {len(df)} rows, saved at {csv_path} ──")
    display_cols = ["question_type", "ground_truth", "prediction",
                    "is_correct_rough", "method", "uncertainty"]
    try:
        from IPython.display import display
        display(df[display_cols].head(15))
    except:
        print(df[display_cols].head(15).to_string())

    # Per-type rough accuracy
    print("\n── Rough accuracy by type (quick check) ──")
    summary = df.groupby("question_type")["is_correct_rough"].agg(
        correct="sum", total="count"
    )
    summary["rough_%"] = (summary["correct"] / summary["total"] * 100).round(1)
    print(summary.to_string())

    # Zero-shot vs RoSE
    print("\n── Zero-shot vs RoSE rough accuracy ──")
    ms = df.groupby("method")["is_correct_rough"].agg(
        correct="sum", total="count"
    )
    ms["rough_%"] = (ms["correct"] / ms["total"] * 100).round(1)
    print(ms.to_string())


## CELL 11 — MERGE ALL THREE TEAMMATES' RESULTS (run by Team Lead)


In [ ]:
# Run this ONLY after all three teammates have saved their files.

def merge_and_evaluate():
    """
    Merge Factoid + Multi Choice + Hypothetical results,
    save a combined JSON + CSV, and run the official evaluation.
    Run this ONLY after all three teammates have saved their files.
    """
    import pandas as pd

    files = {
        "Factoid":      f"{WORK_DIR}/results_factoid.json",
        "Multi Choice": f"{WORK_DIR}/results_multi_choice.json",
        "Hypothetical": f"{WORK_DIR}/results_hypothetical.json",
    }

    combined = []
    for qtype, fpath in files.items():
        if not Path(fpath).exists():
            print(f"⚠  Missing: {fpath}  — skipping")
            continue
        with open(fpath) as f:
            subset = json.load(f)
        print(f"  Loaded {len(subset):>4} records ← {qtype}")
        combined.extend(subset)

    if not combined:
        print("No results to merge."); return

    # ── Save merged JSON (required for official eval script) ──
    merged_json = f"{WORK_DIR}/results_ALL.json"
    with open(merged_json, "w") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)
    print(f"\n✓ Merged JSON → {merged_json}")

    # ── Save merged CSV (human-readable master file) ──
    merged_csv = f"{WORK_DIR}/results_ALL.csv"
    df = save_csv(combined, merged_csv)
    print(f"✓ Merged CSV  → {merged_csv}")

    # ── Display summary table in Colab ──
    print(f"\n  Total records merged: {len(combined)}")
    print("\n── Rough accuracy by type (before official scoring) ──")
    summary = df.groupby("question_type")["is_correct_rough"].agg(
        correct="sum", total="count"
    )
    summary["rough_%"] = (summary["correct"] / summary["total"] * 100).round(1)
    print(summary.to_string())

    # ── Zero-shot vs RoSE breakdown ──
    print("\n── Method breakdown ──")
    method_summary = df.groupby(["question_type", "method"])["is_correct_rough"].agg(
        correct="sum", total="count"
    )
    method_summary["rough_%"] = (method_summary["correct"] / method_summary["total"] * 100).round(1)
    print(method_summary.to_string())

    # ── Official evaluation ──
    print()
    official_evaluate(combined)

    # ── Show first 10 rows as a preview ──
    print("\n── Preview: first 10 rows of merged CSV ──")
    display_cols = ["question_type", "ground_truth", "prediction",
                    "is_correct_rough", "method", "uncertainty"]
    try:
        from IPython.display import display
        display(df[display_cols].head(10))
    except:
        print(df[display_cols].head(10).to_string())

# Uncomment when all three files exist:
# merge_and_evaluate()


## CELL 12 — PUSH RESULTS TO GITHUB (optional)


In [ ]:

def push_to_github(token, github_user="ktahsinr", repo="RoSE"):
    import shutil, subprocess

    repo_dir = f"{WORK_DIR}/RoSE_repo"
    if not Path(repo_dir).exists():
        subprocess.run(["git", "clone",
                        f"https://{token}@github.com/{github_user}/{repo}.git",
                        repo_dir], check=True)

    # Copy results
    dest = Path(repo_dir) / "chartqapro_results"
    dest.mkdir(exist_ok=True)
    shutil.copy(OUTPUT_FILE, dest / Path(OUTPUT_FILE).name)

    os.chdir(repo_dir)
    subprocess.run(["git", "config", "user.email", "team@nsu.edu"])
    subprocess.run(["git", "config", "user.name",  "RoSE-Team"])
    subprocess.run(["git", "add", "."])
    subprocess.run(["git", "commit", "-m",
                    f"Add ChartQAPro results: {ASSIGNED_TYPE}"])
    subprocess.run(["git", "push", "origin", "main"])
    os.chdir(WORK_DIR)
    print(f"✓ Results pushed to GitHub")

# Usage:
# push_to_github("YOUR_GITHUB_TOKEN_HERE")
